# AHS-KT × ASSIST2012 原始 ZIP 最小可运行 Notebook

这本 Notebook 的目标是：

1. 使用 `/root/autodl-tmp/ahs-kt` 项目代码；
2. 使用你指定的 ASSIST2012 原始压缩包：`/root/autodl-tmp/ahs-kt/data/assist2012/2012-2013-data-with-predictions-4-final.zip`；
3. 从 **raw zip → 解压 csv → 构建 AHS-KT bundle → 训练 → 测试评估** 完整跑通；
4. 输出 `acc / auc / f1`。

## 为什么这次不直接复用仓库现成 `.npz`

仓库里确实已经有现成的 `assist2012_train_ahskt.npz` 等文件。

但从第一性原理看，这次你的输入是一个明确的 **原始 zip 路径**，所以最短且正确的路径不是“假装没看到 raw 数据”，而是：
- 先从这个 zip 解压出 csv；
- 再调用项目已有的 `build_assist2012_ahskt.py` 构建脚本；
- 然后用生成出来的 notebook 专用配置训练 `AHSKTModel`；
- 最后补算 `f1`。

## 一个环境层面的注意点

我在真实验证这条链路时，遇到了 OpenBLAS / OMP 线程相关崩溃。

因此这本 Notebook 会在最前面把这些线程环境变量固定成 `1`：
- `OMP_NUM_THREADS`
- `OPENBLAS_NUM_THREADS`
- `MKL_NUM_THREADS`
- `NUMEXPR_NUM_THREADS`
- `VECLIB_MAXIMUM_THREADS`
- `GOTO_NUM_THREADS`

这样能稳定把整条链路跑通。


In [1]:
from pathlib import Path
import os
import sys
import json
import time
import random

PROJECT_ROOT = Path('/root/autodl-tmp/ahs-kt')
SRC_ROOT = PROJECT_ROOT / 'src'
RAW_ZIP_PATH = PROJECT_ROOT / 'data/assist2012/2012-2013-data-with-predictions-4-final.zip'
EXTRACT_DIR = PROJECT_ROOT / 'data/assist2012_raw'
EXTRACTED_CSV_PATH = EXTRACT_DIR / '2012-2013-data-with-predictions-4-final.csv'
DIMKT_DATA_DIR = Path('/root/autodl-tmp/DIMKT/data')

THREAD_ENV = {
    'OMP_NUM_THREADS': '1',
    'OPENBLAS_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
    'NUMEXPR_NUM_THREADS': '1',
    'VECLIB_MAXIMUM_THREADS': '1',
    'GOTO_NUM_THREADS': '1',
}
for key, value in THREAD_ENV.items():
    os.environ[key] = value

assert PROJECT_ROOT.exists(), f'找不到项目目录: {PROJECT_ROOT}'
assert RAW_ZIP_PATH.exists(), f'找不到原始 zip: {RAW_ZIP_PATH}'
assert DIMKT_DATA_DIR.exists(), f'找不到 DIMKT 参考目录: {DIMKT_DATA_DIR}'

os.chdir(PROJECT_ROOT)
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RAW_ZIP_PATH =', RAW_ZIP_PATH)
print('DIMKT_DATA_DIR =', DIMKT_DATA_DIR)
print('EXTRACT_DIR =', EXTRACT_DIR)
print('线程环境变量 =')
for key in THREAD_ENV:
    print(f'  {key}={os.environ[key]}')


PROJECT_ROOT = /root/autodl-tmp/ahs-kt
RAW_ZIP_PATH = /root/autodl-tmp/ahs-kt/data/assist2012/2012-2013-data-with-predictions-4-final.zip
DIMKT_DATA_DIR = /root/autodl-tmp/DIMKT/data
EXTRACT_DIR = /root/autodl-tmp/ahs-kt/data/assist2012_raw
线程环境变量 =
  OMP_NUM_THREADS=1
  OPENBLAS_NUM_THREADS=1
  MKL_NUM_THREADS=1
  NUMEXPR_NUM_THREADS=1
  VECLIB_MAXIMUM_THREADS=1
  GOTO_NUM_THREADS=1


In [2]:
import subprocess

import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

from ahskt.config import load_config
from ahskt.data.dataset import load_bundle_from_config
from ahskt.models.ahs_kt import AHSKTModel
from ahskt.training.engine import fit_and_evaluate

print('TensorFlow version =', tf.__version__)
print('GPU devices =', tf.config.list_physical_devices('GPU'))
for gpu_device in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu_device, True)
    except RuntimeError:
        pass


TensorFlow version = 2.8.0
GPU devices = [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 参数区

这本 Notebook 默认会：
- 强制从 raw zip 重新解压；
- 重新调用项目构建脚本生成 notebook 专用 bundle；
- 用项目默认的 ASSIST2012 AHS-KT 训练超参数训练 5 个 epoch；
- 最后额外补算 `f1`。

如果你只是想复用已经生成过的中间文件，可以把下面的 `FORCE_REBUILD` 改成 `False`。


In [3]:
SEED = 2026
FORCE_REBUILD = True
TASK_NAME = 'ahskt_assist2012_from_zip_notebook'

NOTEBOOK_DATA_DIR = PROJECT_ROOT / 'data/assist2012_from_zip_notebook'
NOTEBOOK_METADATA_PATH = NOTEBOOK_DATA_DIR / 'assist2012_metadata.json'
NOTEBOOK_CONFIG_PATH = PROJECT_ROOT / 'configs/ahskt_assist2012_from_zip_notebook.json'
NOTEBOOK_OUTPUT_ROOT = PROJECT_ROOT / 'outputs/assist2012_from_zip_notebook_run'
NOTEBOOK_METRICS_PATH = NOTEBOOK_OUTPUT_ROOT / f'{TASK_NAME}_metrics.json'
NOTEBOOK_METRICS_WITH_F1_PATH = NOTEBOOK_OUTPUT_ROOT / f'{TASK_NAME}_metrics_with_f1.json'

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TASK_NAME =', TASK_NAME)
print('NOTEBOOK_DATA_DIR =', NOTEBOOK_DATA_DIR)
print('NOTEBOOK_CONFIG_PATH =', NOTEBOOK_CONFIG_PATH)
print('NOTEBOOK_OUTPUT_ROOT =', NOTEBOOK_OUTPUT_ROOT)


TASK_NAME = ahskt_assist2012_from_zip_notebook
NOTEBOOK_DATA_DIR = /root/autodl-tmp/ahs-kt/data/assist2012_from_zip_notebook
NOTEBOOK_CONFIG_PATH = /root/autodl-tmp/ahs-kt/configs/ahskt_assist2012_from_zip_notebook.json
NOTEBOOK_OUTPUT_ROOT = /root/autodl-tmp/ahs-kt/outputs/assist2012_from_zip_notebook_run


In [4]:
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

unzip_cmd = [
    'unzip', '-o', str(RAW_ZIP_PATH), '-d', str(EXTRACT_DIR),
]
print('执行解压命令:')
print(' '.join(unzip_cmd))
subprocess.run(unzip_cmd, check=True)

assert EXTRACTED_CSV_PATH.exists(), f'解压后仍找不到 csv: {EXTRACTED_CSV_PATH}'
print('解压完成，csv 路径 =', EXTRACTED_CSV_PATH)
print('csv 大小(GB) =', round(EXTRACTED_CSV_PATH.stat().st_size / (1024 ** 3), 3))


执行解压命令:
unzip -o /root/autodl-tmp/ahs-kt/data/assist2012/2012-2013-data-with-predictions-4-final.zip -d /root/autodl-tmp/ahs-kt/data/assist2012_raw
Archive:  /root/autodl-tmp/ahs-kt/data/assist2012/2012-2013-data-with-predictions-4-final.zip
  inflating: /root/autodl-tmp/ahs-kt/data/assist2012_raw/2012-2013-data-with-predictions-4-final.csv  
解压完成，csv 路径 = /root/autodl-tmp/ahs-kt/data/assist2012_raw/2012-2013-data-with-predictions-4-final.csv
csv 大小(GB) = 2.803


In [5]:
build_env = os.environ.copy()
build_env.update(THREAD_ENV)

build_cmd = [
    sys.executable,
    'scripts/build_assist2012_ahskt.py',
    '--csv-path', str(EXTRACTED_CSV_PATH),
    '--dimkt-data-dir', str(DIMKT_DATA_DIR),
    '--output-dir', str(NOTEBOOK_DATA_DIR),
    '--config-output', str(NOTEBOOK_CONFIG_PATH),
    '--task-name', TASK_NAME,
    '--outputs-root', str(NOTEBOOK_OUTPUT_ROOT.relative_to(PROJECT_ROOT)),
]

if FORCE_REBUILD or (not NOTEBOOK_CONFIG_PATH.exists()) or (not NOTEBOOK_METADATA_PATH.exists()):
    print('执行构建命令:')
    print(' '.join(build_cmd))
    build_result = subprocess.run(
        build_cmd,
        check=True,
        cwd=PROJECT_ROOT,
        env=build_env,
        text=True,
        capture_output=True,
    )
    print(build_result.stdout)
else:
    print('检测到 notebook 专用 bundle 已存在，跳过重建。')

assert NOTEBOOK_CONFIG_PATH.exists(), f'构建后找不到配置文件: {NOTEBOOK_CONFIG_PATH}'
assert NOTEBOOK_METADATA_PATH.exists(), f'构建后找不到元数据: {NOTEBOOK_METADATA_PATH}'


执行构建命令:
/root/miniconda3/bin/python scripts/build_assist2012_ahskt.py --csv-path /root/autodl-tmp/ahs-kt/data/assist2012_raw/2012-2013-data-with-predictions-4-final.csv --dimkt-data-dir /root/autodl-tmp/DIMKT/data --output-dir /root/autodl-tmp/ahs-kt/data/assist2012_from_zip_notebook --config-output /root/autodl-tmp/ahs-kt/configs/ahskt_assist2012_from_zip_notebook.json --task-name ahskt_assist2012_from_zip_notebook --outputs-root outputs/assist2012_from_zip_notebook_run
{
  "train_path": "/root/autodl-tmp/ahs-kt/data/assist2012_from_zip_notebook/assist2012_train_ahskt.npz",
  "valid_path": "/root/autodl-tmp/ahs-kt/data/assist2012_from_zip_notebook/assist2012_valid_ahskt.npz",
  "test_path": "/root/autodl-tmp/ahs-kt/data/assist2012_from_zip_notebook/assist2012_test_ahskt.npz",
  "metadata_path": "/root/autodl-tmp/ahs-kt/data/assist2012_from_zip_notebook/assist2012_metadata.json",
  "config_output": "/root/autodl-tmp/ahs-kt/configs/ahskt_assist2012_from_zip_notebook.json",
  "split_summ

In [6]:
with open(NOTEBOOK_METADATA_PATH, 'r', encoding='utf-8') as f:
    assist2012_metadata = json.load(f)

with open(NOTEBOOK_CONFIG_PATH, 'r', encoding='utf-8') as f:
    notebook_config_payload = json.load(f)

print('metadata =')
print(json.dumps(assist2012_metadata, ensure_ascii=False, indent=2))
print()
print('config =')
print(json.dumps(notebook_config_payload, ensure_ascii=False, indent=2))


metadata =
{
  "dataset_name": "assist2012",
  "sequence_length": 100,
  "num_questions": 53091,
  "num_concepts": 265,
  "num_question_difficulty": 101,
  "num_concept_difficulty": 98,
  "num_behavior_clusters": 5,
  "num_filtered_interactions": 2397617,
  "num_users": 28382,
  "clip_values": {
    "attempts": 6.0,
    "hints": 4.0,
    "speed": 20.95703887939453
  },
  "behavior_centers": [
    {
      "cluster_id": 1,
      "center_attempts_log": 0.6935508251190186,
      "center_hints_log": 0.004999933298677206,
      "center_speed_log": 0.8267138600349426,
      "center_attempts_raw": 1.0008074045181274,
      "center_hints_raw": 0.0050124539993703365,
      "center_speed_qpm_raw": 1.285794973373413
    },
    {
      "cluster_id": 2,
      "center_attempts_log": 0.7027933597564697,
      "center_hints_log": 0.026431677863001823,
      "center_speed_log": 2.0710513591766357,
      "center_attempts_raw": 1.0193856954574585,
      "center_hints_raw": 0.02678409405052662,
      "cent

In [7]:
config = load_config(NOTEBOOK_CONFIG_PATH, project_root=PROJECT_ROOT)
train_bundle, valid_bundle, test_bundle = load_bundle_from_config(config)

print('train / valid / test 序列数 =', train_bundle.num_samples, valid_bundle.num_samples, test_bundle.num_samples)
print('sequence_length =', train_bundle.sequence_length)
print('train bundle shapes:')
for key, value in train_bundle.as_dict().items():
    print(f'  {key}: {value.shape}')


train / valid / test 序列数 = 26867 6709 8452
sequence_length = 100
train bundle shapes:
  question_ids: (26867, 100)
  concept_ids: (26867, 100)
  responses: (26867, 100)
  question_difficulty: (26867, 100)
  concept_difficulty: (26867, 100)
  attempts: (26867, 100)
  hints: (26867, 100)
  speed: (26867, 100)
  behavior_cluster: (26867, 100)
  mask: (26867, 100)
  question_easiness: (26867, 100)
  concept_easiness: (26867, 100)
  question_confidence: (26867, 100)
  concept_confidence: (26867, 100)


## 开始训练 AHS-KT

下面这段逻辑和项目脚本 `scripts/train_ahskt.py` 基本一致：
- `load_config`
- `load_bundle_from_config`
- `AHSKTModel`
- `fit_and_evaluate`

区别只在于：
- 这里直接在 Notebook 里执行，方便查看中间对象；
- 训练结束后会继续补算 `f1`。


In [8]:
model = AHSKTModel(config.model)

train_start = time.time()
metrics_summary = fit_and_evaluate(
    model=model,
    train_bundle=train_bundle,
    valid_bundle=valid_bundle,
    test_bundle=test_bundle,
    config=config,
)
train_elapsed = time.time() - train_start

NOTEBOOK_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_METRICS_PATH.write_text(json.dumps(metrics_summary, ensure_ascii=False, indent=2), encoding='utf-8')

print('训练耗时(秒) =', round(train_elapsed, 2))
print('NOTEBOOK_METRICS_PATH =', NOTEBOOK_METRICS_PATH)
print(json.dumps(metrics_summary, ensure_ascii=False, indent=2))


2026-04-07 11:14:09.792794: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-07 11:14:10.658112: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1525] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22182 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:57:00.0, compute capability: 8.6
2026-04-07 11:14:12.455005: I tensorflow/stream_executor/cuda/cuda_blas.cc:1786] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2026-04-07 11:14:13.281925: I tensorflow/stream_executor/cuda/cuda_dnn.cc:368] Loaded cuDNN version 8200


训练耗时(秒) = 124.17
NOTEBOOK_METRICS_PATH = /root/autodl-tmp/ahs-kt/outputs/assist2012_from_zip_notebook_run/ahskt_assist2012_from_zip_notebook_metrics.json
{
  "task_name": "ahskt_assist2012_from_zip_notebook",
  "seed": 2026,
  "best_epoch": 3,
  "best_valid_auc": 0.7171843483971859,
  "test_metrics": {
    "loss": 0.5416816473007202,
    "auc": 0.7199659841202692,
    "acc": 0.7372733010730065,
    "rmse": 0.42531904578208923
  },
  "history": [
    {
      "epoch": 1,
      "train": {
        "loss": 0.5593898296356201,
        "auc": 0.6880087100432163,
        "acc": 0.7277543598754455,
        "rmse": 0.4334467351436615
      },
      "valid": {
        "loss": 0.5445663928985596,
        "auc": 0.7080283542803119,
        "acc": 0.737696362958624,
        "rmse": 0.4263812303543091
      }
    },
    {
      "epoch": 2,
      "train": {
        "loss": 0.5388464331626892,
        "auc": 0.7219394157853104,
        "acc": 0.7397515825373537,
        "rmse": 0.42389169335365295
    

## 补充 F1 指标

项目原生输出默认没有 `f1`。

这里沿用同一份测试集预测，额外补出：
- `acc`
- `auc`
- `f1`


In [9]:
def collect_targets_and_predictions(model, bundle, batch_size):
    dataset = bundle.to_tf_dataset(batch_size=batch_size, shuffle=False)
    all_targets = []
    all_predictions = []
    for batch in dataset:
        logits = model(batch, training=False)
        next_logits = logits[:, :-1]
        next_targets = tf.cast(batch['responses'][:, 1:], tf.float32)
        next_mask = tf.cast(batch['mask'][:, 1:], tf.float32)
        valid_logits = tf.boolean_mask(next_logits, next_mask > 0)
        valid_targets = tf.boolean_mask(next_targets, next_mask > 0)
        all_targets.append(valid_targets.numpy())
        all_predictions.append(tf.sigmoid(valid_logits).numpy())
    return np.concatenate(all_targets, axis=0), np.concatenate(all_predictions, axis=0)


test_targets, test_predictions = collect_targets_and_predictions(
    model=model,
    bundle=test_bundle,
    batch_size=config.training.batch_size,
)

test_binary_predictions = (test_predictions > 0.5).astype(int)
summary_with_f1 = {
    'acc': float(accuracy_score(test_targets, test_binary_predictions)),
    'auc': float(roc_auc_score(test_targets, test_predictions)),
    'f1': float(f1_score(test_targets, test_binary_predictions)),
    'loss': float(metrics_summary['test_metrics']['loss']),
    'rmse': float(metrics_summary['test_metrics']['rmse']),
    'num_test_points': int(len(test_targets)),
}

NOTEBOOK_METRICS_WITH_F1_PATH.write_text(
    json.dumps(summary_with_f1, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('NOTEBOOK_METRICS_WITH_F1_PATH =', NOTEBOOK_METRICS_WITH_F1_PATH)
print(json.dumps(summary_with_f1, ensure_ascii=False, indent=2))


NOTEBOOK_METRICS_WITH_F1_PATH = /root/autodl-tmp/ahs-kt/outputs/assist2012_from_zip_notebook_run/ahskt_assist2012_from_zip_notebook_metrics_with_f1.json
{
  "acc": 0.7372733010730065,
  "auc": 0.7199659841202692,
  "f1": 0.8312559120023784,
  "loss": 0.5416816473007202,
  "rmse": 0.42531904578208923,
  "num_test_points": 475300
}


## 结论

这本 Notebook 已经把以下链路串起来了：
- 原始数据：`assist2012` raw zip
- 数据构建：项目原生 `build_assist2012_ahskt.py`
- 模型：`AHSKTModel`
- 训练：项目原生 `fit_and_evaluate`
- 结果：输出 `acc / auc / f1`

如果你后面还想继续做更正式的实验，下一步最自然的是：
- 跑多随机种子；
- 对比 `assist2012_v1 / v2 / ablation`；
- 汇总均值与方差。
